#  Extract satellite data over drone sites

This will form the basis of the canopy height training data

## Import libraries

In [ ]:
import os
import odc.stac
import pystac_client
import pandas as pd
import xarray as xr
import rioxarray as rxr
import numpy as np
import matplotlib.pyplot as plt

from odc.geo import BoundingBox
from odc.geo.xr import assign_crs
from odc.stac import stac_load, configure_s3_access

from odc.algo import geomedian_with_mads
from dea_tools.dask import create_local_dask_cluster
from dea_tools.bandindices import calculate_indices
from odc.stac import load, configure_s3_access

from dualpol_indices import dualpol_indices
from speckle_filters import apply_lee_filter

## Start dask client and connect to STAC

In [ ]:
#search catalog
catalog = pystac_client.Client.open("https://explorer.dea.ga.gov.au/stac")
dev_stac_catalog = pystac_client.Client.open("https://explorer.dev.dea.ga.gov.au/stac")

In [ ]:
client = create_local_dask_cluster(return_client=True)

In [ ]:
configure_s3_access(cloud_defaults=True, aws_unsigned=True)

## Analysis Parameters

In [ ]:
# Set a start and end date
start_date = "2025-01-01"
end_date = "2025-08-31"

# data loading params
crs="EPSG:3577"
resolution=20
measurements=[
 'nbart_red',
 'nbart_blue',
 'nbart_green',
 'nbart_nir_1',
 'nbart_nir_2',
 'nbart_swir_2',
 'nbart_swir_3',
 'nbart_red_edge_1',
 'nbart_red_edge_2',
 'nbart_red_edge_3',
]

masking_band = 'oa_s2cloudless_mask'

## Loop through canopy height data and extracting training data

In [ ]:
chm_tifs = [i for i in os.listdir('results/chm_tifs') if i.endswith('tif')]

In [ ]:
%%time
for tif in chm_tifs:
    
    name = tif.split('.')[0]
    print(name)

    # open canopy height data
    chm = rxr.open_rasterio(f'results/chm_tifs/{tif}').squeeze().drop_vars('band')
    bbox=chm.odc.geobox.to_crs('EPSG:4326').boundingbox

    # get S2 data
    query = catalog.search(
        bbox=bbox,
        collections=['ga_s2am_ard_3', 'ga_s2bm_ard_3'],
        datetime=f"{start_date}/{end_date}"
        )
    
    items = list(query.items())

    ds = odc.stac.load(
        items,
        bands=measurements+[masking_band],
        crs=crs,
        resolution=resolution,
        groupby="solar_day",
        bbox=bbox,
        chunks={},
    )
    
    clear = xr.where(ds['oa_s2cloudless_mask'] == 1, True, False)
    ds = ds.drop_vars(masking_band).where(clear)

    # calculate geomedians
    s2_gm = geomedian_with_mads(
        ds,
        reshape_strategy='yxbt',
        compute_mads=True 
    )
    
    # bring into memory
    s2_gm = assign_crs(s2_gm.load(), crs=crs)
    s2_gm = s2_gm.drop_vars('count')

    # add veg indices
    s2_gm = calculate_indices(s2_gm,index=['EVI', 'kNDVI'], drop=False,collection='ga_s2_3')

    #------------------------
    # Load Sentinel-1 
    s1_items = dev_stac_catalog.search(
        bbox=bbox,
        collections=["ga_s1_nrb_iw_vv_vh_0"],
        datetime=f"{start_date}/{end_date}"
    )

    s1_items = list(s1_items.items())
    
    ds_s1 = odc.stac.load(
        s1_items,
        bands=["VV_gamma0", "VH_gamma0", "mask"],
        like=s2_gm.odc.geobox, #match boundary of gm
        groupby="solar_day",
        chunks={}
    )

    ds_s1 = ds_s1.rename({'VV_gamma0':'VV', 'VH_gamma0':'VH'})
    
    # lee filtering
    ds_s1["VV"] = apply_lee_filter(ds_s1["VV"], size=4)
    ds_s1["VH"] = apply_lee_filter(ds_s1["VH"], size=4)
    
    valid_mask = xr.where(ds_s1.mask>0, False, True)
    ds_s1 = ds_s1.where(valid_mask).drop_vars('mask')

    #calculate Radar veg index
    rvi = dualpol_indices(ds_s1, index=["RVI"], drop=True)

    # Create temporal summary stats on S1
    def dB_scale(data): 
        '''Scales a xarray.DataArray with linear DN to a dB scale.'''
        # Explicitly set negative data to nan to avoid log of negative number
        negative_free_data = data.where(data >= 0, np.nan)
        return 10 * np.log10(negative_free_data)

    # Scale to plot data in decibels
    ds_s1["VH"] = dB_scale(ds_s1.VH)
    ds_s1["VV"] = dB_scale(ds_s1.VV)
    ds_s1["VV-VH"] = dB_scale(ds_s1.VV - ds_s1.VH)
    
    ds_s1 = ds_s1.where(valid_mask)
    
    #take temporal stats
    ds_s1_mean = ds_s1.mean('time')
    rvi_mean = rvi.median('time')
    # ds_s1_max = ds_s1.max('time')
    # ds_s1_min = ds_s1.min('time')
    # ds_s1_std = ds_s1.std('time')
    
    ds_s1_mean = ds_s1_mean.rename({'VV':'VV_mean', 'VH':'VH_mean', 'VV-VH':'VV-VH_mean'})
    # ds_s1_max = ds_s1_max.rename({'VV':'VV_max', 'VH':'VH_max', 'VV-VH':'VV-VH_max'})
    # ds_s1_min = ds_s1_min.rename({'VV':'VV_min', 'VH':'VH_min', 'VV-VH':'VV-VH_min'})
    # ds_s1_std = ds_s1_std.rename({'VV':'VV_std', 'VH':'VH_std', 'VV-VH':'VV-VH_std'})

    # ----combine data-----------------
    combined = xr.merge([s2_gm, ds_s1_mean, rvi_mean], compat='override').compute()
    # combined = s2_gm.copy()

    #reproject CHM to match satellite data, and add to combined arrays
    chm = chm.odc.reproject(how=combined.odc.geobox, resampling='average', dst_nodata=np.nan)
    chm_mask = ~np.isnan(chm)
    
    combined = combined.where(chm_mask)
    combined['canopy_height'] = chm

    # Now export results to disk as a csv
    df = combined.drop_vars('spatial_ref').to_dataframe().dropna(how='any').reset_index(drop=True)
    df.to_csv(f'results/training_data/{name}.csv')

## Combine all outputs into one training set

In [ ]:
# Path to your folder
folder_path = "results/training_data/"

# List all CSV files in the folder
csv_files = glob.glob(os.path.join(folder_path, "*.csv"))

# Read and concatenate them into one DataFrame
df = pd.concat((pd.read_csv(f) for f in csv_files))
df = df.drop('Unnamed: 0', axis=1)

df.replace([np.inf, -np.inf], np.nan, inplace=True)
df = df.dropna(how='any')
df.head()

In [ ]:
df.to_csv('results/CH_combined_training_data.csv')